# GCN Example 3 (PyTorch) — Link Prediction (Graph Auto-Encoder)

PyTorch version of `link_prediction.ipynb`. We hide 15% of the Karate Club's edges and
train a **Graph Auto-Encoder** to recover them.

- **Encoder**: a 2-layer GCN turns each node into an embedding $z_i$.
- **Decoder**: the score of edge $(i, j)$ is $\sigma(z_i^\top z_j)$ — similar embeddings
  mean a likely edge.
- **Training**: binary cross-entropy contrasting real edges (label 1) with sampled
  non-edges (label 0).
- **Metric**: AUC on held-out edges vs. fake edges.

> Self-contained, runs in seconds on CPU. Needs `torch`.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

## 1. Graph and train/test edge split

Hold out 15% of edges as test positives; build the message-passing adjacency from the
training edges only.

In [ ]:
edges = [
    (0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(0,7),(0,8),(0,10),(0,11),(0,12),
    (0,13),(0,17),(0,19),(0,21),(0,31),(1,2),(1,3),(1,7),(1,13),(1,17),(1,19),
    (1,21),(1,30),(2,3),(2,7),(2,8),(2,9),(2,13),(2,27),(2,28),(2,32),(3,7),
    (3,12),(3,13),(4,6),(4,10),(5,6),(5,10),(5,16),(6,16),(8,30),(8,32),(8,33),
    (9,33),(13,33),(14,32),(14,33),(15,32),(15,33),(18,32),(18,33),(19,33),
    (20,32),(20,33),(22,32),(22,33),(23,25),(23,27),(23,29),(23,32),(23,33),
    (24,25),(24,27),(24,31),(25,31),(26,29),(26,33),(27,33),(28,31),(28,33),
    (29,32),(29,33),(30,32),(30,33),(31,32),(31,33),(32,33),
]
N = 34
all_edges = [(min(i, j), max(i, j)) for i, j in edges]
edge_set = set(all_edges)

order = np.random.permutation(len(all_edges))
n_test = int(0.15 * len(all_edges))
test_pos = [all_edges[i] for i in order[:n_test]]
train_pos = [all_edges[i] for i in order[n_test:]]

A = np.zeros((N, N), dtype='float32')
for i, j in train_pos:
    A[i, j] = A[j, i] = 1.0
A_tilde = A + np.eye(N, dtype='float32')
d_inv_sqrt = np.diag(1.0 / np.sqrt(A_tilde.sum(1)))
A_hat = torch.tensor((d_inv_sqrt @ A_tilde @ d_inv_sqrt).astype('float32'), device=device)
print(f'edges: {len(all_edges)} | train: {len(train_pos)} | test: {len(test_pos)}')

In [ ]:
def sample_negatives(k):
    negs, seen = [], set()
    while len(negs) < k:
        i, j = np.random.randint(N), np.random.randint(N)
        if i == j:
            continue
        a, b = min(i, j), max(i, j)
        if (a, b) in edge_set or (a, b) in seen:
            continue
        seen.add((a, b)); negs.append((a, b))
    return np.array(negs)

def auc_score(pos_scores, neg_scores):
    s = np.concatenate([pos_scores, neg_scores])
    order = np.argsort(s)
    ranks = np.empty(len(s)); ranks[order] = np.arange(1, len(s) + 1)
    n_pos, n_neg = len(pos_scores), len(neg_scores)
    return (ranks[:n_pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

## 2. GCN encoder + dot-product decoder

In [ ]:
class GAE(nn.Module):
    def __init__(self, in_features=N, hidden=32, emb=16):
        super().__init__()
        self.lin1 = nn.Linear(in_features, hidden, bias=False)
        self.lin2 = nn.Linear(hidden, emb, bias=False)

    def encode(self, X, A_hat):
        h = F.relu(A_hat @ self.lin1(X))
        return A_hat @ self.lin2(h)

    def decode(self, z, pairs):
        zi = z[pairs[:, 0]]
        zj = z[pairs[:, 1]]
        return (zi * zj).sum(dim=1)   # dot-product logits

X = torch.eye(N, device=device)
model = GAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
train_pos_t = torch.tensor(np.array(train_pos), device=device)

## 3. Train

Fresh negatives each epoch; binary cross-entropy on logits via `binary_cross_entropy_with_logits`.

In [ ]:
EPOCHS = 200
for epoch in range(1, EPOCHS + 1):
    model.train()
    neg = torch.tensor(sample_negatives(len(train_pos)), device=device)
    z = model.encode(X, A_hat)
    pos_logit = model.decode(z, train_pos_t)
    neg_logit = model.decode(z, neg)
    loss = (F.binary_cross_entropy_with_logits(pos_logit, torch.ones_like(pos_logit)) +
            F.binary_cross_entropy_with_logits(neg_logit, torch.zeros_like(neg_logit)))
    optimizer.zero_grad(); loss.backward(); optimizer.step()

    if epoch % 20 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            z = model.encode(X, A_hat)
            test_neg = torch.tensor(sample_negatives(len(test_pos)), device=device)
            pos_s = model.decode(z, torch.tensor(np.array(test_pos), device=device)).cpu().numpy()
            neg_s = model.decode(z, test_neg).cpu().numpy()
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={loss.item():.4f}  test_AUC={auc_score(pos_s, neg_s):.3f}')

## 4. Inspect predictions

In [ ]:
model.eval()
with torch.no_grad():
    z = model.encode(X, A_hat)
    test_neg = torch.tensor(sample_negatives(len(test_pos)), device=device)
    pos_prob = torch.sigmoid(model.decode(z, torch.tensor(np.array(test_pos), device=device))).cpu().numpy()
    neg_prob = torch.sigmoid(model.decode(z, test_neg)).cpu().numpy()
print(f'Final test AUC: {auc_score(pos_prob, neg_prob):.3f}\n')
print(f'Mean prob on REAL held-out edges: {pos_prob.mean():.3f}')
print(f'Mean prob on FAKE edges:          {neg_prob.mean():.3f}')

### Things to try

- Increase the embedding size or add a layer.
- Replace the dot-product decoder with an MLP on `[z_i || z_j]`.
- Average AUC over several random splits.
- Add a KL term to make a **variational** GAE (VGAE).